# Environment probe — Weng et al. (2024) reproduction

Builds the exact stack the five model kernels use and runs **one epoch on one
segment** for each of the authors' five architectures, then stops.

**The numbers this prints are meaningless.** The only question it answers is
whether the authors' code executes in this environment — worth answering in
minutes rather than discovering hours into a real sweep.

Run this first. When every architecture reports OK, push the model kernels.

## Three documented deviations, all forced

**1. Python version.** Kaggle's image is Python 3.12. The authors pin
`torch==2.1.2`, which has no cp312 wheel — torch added 3.12 support in 2.2.0 — so
their stack cannot install on the stock image. Rather than bump torch and lose
fidelity, these kernels use `uv` to fetch a standalone **Python 3.11** and build
the authors' exact stack inside it.

**2. pandas.** `torch_geometric_temporal==0.54.0` declares `pandas<=1.3.5`,
contradicting the authors' own `pandas~=2.2.0`; pip answers
`ResolutionImpossible`. → PGT installed with `--no-deps`, honouring the authors'
pandas pin. Only PGT's stale transitive pin is ignored, which the authors had
already overridden.

**3. torch_geometric.** The pinned pair `torch_geometric==2.5.3` +
`torch_geometric_temporal==0.54.0` **cannot import**:

```
File ".../torch_geometric_temporal/nn/attention/tsagcn.py", line 6
    from torch_geometric.utils.to_dense_adj import to_dense_adj
ModuleNotFoundError: No module named 'torch_geometric.utils.to_dense_adj'
```

PyG moved that module in 2.4. Tested: 2.4.0 is the newest release where all five
architectures import. PGT stays at the authors' `0.54.0`, because PGT defines the
model architectures — changing it risks changing the models.

Everything else — `torch==2.1.2`, `numpy~=1.26.2`, `pandas~=2.2.0`,
`scikit_learn==1.4.0`, `statsmodels==0.14.1` — is exactly what the authors named.
All three deviations must be restated wherever these numbers are cited. See
`reproduction/REPRODUCIBILITY_MATRIX.md`.

## What the authors' code actually does

Read this before the results — it is what the numbers mean. Everything below is
in `Models/evaluation.py` and `Models/gnn_models.py`; nothing is our paraphrase
of intent, only a description of the code that runs.

### Data (`load_data`, `create_dataset_single`)

```python
x = np.nan_to_num(np.load(data_file, allow_pickle=True))
mean, std = np.mean(x[..., -6]), np.std(x[..., -6])      # whole series
x = z_norm(torch.tensor(x), mean, std)
x = x[:int(x.shape[0] * subset), ...]                     # truncate AFTER
```

The array is `(459 weeks, 25 districts, 11 features)`; `x[..., -6]` is index 5,
weekly cases. Note the ordering: statistics are computed over the **entire**
series, including the test period, and only then is the segment truncated.
`inverse_z_norm` reuses those same statistics when scoring.

Windows are built as `window_size=3` past weeks → `predict_ahead=3` future weeks.
For the GNNs the authors pass `use_disease_only=True`, so the models see **only
the case channel** — the ten meteorological covariates are not used.

### Graph

`load_adjacency_matrix` builds a 25×25 binary matrix from
`sri_lanka_adj_list.json` with self-loops, giving 141 directed edges. Districts
are ordered by `sorted()` of the JSON keys.

### Training (`train_model`, `train_single_shot`, ...)

Adam at `lr=1e-4`, `weight_decay=5e-5`, `MSELoss`, `num_epochs=50`,
`batch_size=1`, no early stopping and no validation-based model selection — the
final epoch's weights are the ones scored. Module-level `torch.manual_seed(0)`,
`random.seed(0)` and `np.random.seed(0)` make it deterministic.

### Evaluation (`infer`) — the part that decides what Table I means

```python
rmse += RMSE(truth, pred)
mae  += MAE(truth, pred)
...
rmse /= n
```

Metrics accumulate **per batch** and are divided by the batch count. With
`batch_size=1` that is the mean of per-window RMSEs, not the RMSE of the pooled
predictions — the two differ substantially on a heavy-tailed target.

And in every `run_*`:

```python
y_pred, y_truth, _, _  = infer(model, "cpu", test, m, s, "Test")   # discarded
y_pred, y_truth, ma, rm = infer(model, "cpu", full, m, s, "Full")  # appended
maes += [ma]; rmses += [rm]
```

The held-out test score is computed and thrown away. The number that becomes
Table I's "Cross Validated" column comes from `full` — every window in the
segment, including the 70% the model trained on.

### Cross-validation

`run_*([0.6, 0.7, 0.8, 0.9, 1.0])` reinitializes the model per segment, trains on
the first 70% of that prefix, and reports mean ± population std across the five
segments.

**This notebook changes none of that.** It reproduces the numbers as the authors
produce them. Whether that protocol supports the paper's conclusions is a
separate question, examined in `crosscheck/FINDINGS.md`.

## 1. Environment — the authors' pins, under a standalone Python 3.11

In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. The authors' repository and data, verified by checksum

In [ ]:
if not SRC.exists():
    sh("git", "clone", REPO, str(SRC))
sh("git", "-C", str(SRC), "checkout", "--quiet", COMMIT)

head = sh("git", "-C", str(SRC), "rev-parse", "HEAD", quiet=True).stdout.strip()
assert head == COMMIT, f"expected {COMMIT}, got {head}"
print("repository at", head)

# Verify the released inputs by git blob SHA. This is platform-independent --
# unlike an md5 of the checked-out bytes, which differs between a Windows and a
# Linux clone because git rewrites line endings in text files.
for rel, want in [("Data/Datasets/sri_lanka_2013-2022_shifted.npy", BLOB_NPY),
                  ("Models/sri_lanka_adj_list.json", BLOB_ADJ)]:
    got = sh("git", "-C", str(SRC), "rev-parse", f"HEAD:{rel}", quiet=True).stdout.strip()
    print(("OK  " if got == want else "MISMATCH ") + f"{rel}  {got}")
    assert got == want, f"{rel} blob changed: {got} != {want}"

## 3. Smoke test — one epoch, one segment, all five architectures

`evaluation.num_epochs` is overridden to 1 here and **only** here. The model
kernels leave the authors' `num_epochs = 50` untouched.

In [ ]:
results = {}
for runner in ["run_stgat", "run_a3tgcn", "run_astgcn", "run_dcrnn", "run_aagcn"]:
    script = (
        "import sys\n"
        "sys.path.insert(0, '.')\n"
        "import evaluation\n"
        "evaluation.num_epochs = 1        # smoke test only, NOT a reproduction\n"
        f"evaluation.{runner}([0.6])\n"
    )
    started = time.time()
    r = sh(str(PY311), "-u", "-c", script, cwd=str(SRC / "Models"), check=False, quiet=True)
    ok = r.returncode == 0
    if ok:
        lines = [ln for ln in r.stdout.splitlines() if "Average" in ln]
        detail = lines[-2] if len(lines) >= 2 else ""
    else:
        detail = (r.stdout.strip().splitlines() or ["?"])[-1][:160]
    results[runner] = ok
    print(f"{runner:12s} {'OK  ' if ok else 'FAIL'} {time.time() - started:6.1f}s  {detail}")

print()
if all(results.values()):
    print("ALL FIVE ARCHITECTURES RUN. Safe to push the model kernels.")
else:
    print("BROKEN:", [k for k, v in results.items() if not v])
    raise SystemExit("environment probe failed")